In [7]:
import numpy as np

# Tek kübitin iki taban durumu (basis states)
ket0 = np.array([1, 0], dtype=complex)   # |0⟩
ket1 = np.array([0, 1], dtype=complex)   # |1⟩

print("|0> =", ket0)
print("|1> =", ket1)

# Dört taban durumunun hepsini dene
# İki kübitlik |00⟩ = |0⟩ ⊗ |0⟩
ket00 = np.kron(ket0, ket0)
ket01 = np.kron(ket0, ket1)
ket10 = np.kron(ket1, ket0)
ket11 = np.kron(ket1, ket1)

print("|00> =", ket00)

|0> = [1.+0.j 0.+0.j]
|1> = [0.+0.j 1.+0.j]
|00> = [1.+0.j 0.+0.j 0.+0.j 0.+0.j]


In [4]:
# Pauli-X: bit çevirme (bit flip)
X = np.array([[0, 1],
              [1, 0]], dtype=complex)

# Pauli-Z: faz çevirme (phase flip)
Z = np.array([[1,  0],
              [0, -1]], dtype=complex)

# Hadamard: süperpozisyon yapar
H = np.array([[1,  1],
              [1, -1]], dtype=complex) / np.sqrt(2)

# Ry(θ): y ekseni etrafında θ kadar döndürme (rotation)
def Ry(theta):
    c = np.cos(theta / 2)
    s = np.sin(theta / 2)
    return np.array([[c, -s],
                     [s,  c]], dtype=complex)

# Hızlı kontrol: kapıları |0⟩'a uygula
print("X|0>  =", X @ ket0)
print("Z|0>  =", Z @ ket0)
print("H|0>  =", H @ ket0)
print("Ry(pi)|0> =",np.round(Ry(np.pi) @ ket0, 3))


X|0>  = [0.+0.j 1.+0.j]
Z|0>  = [1.+0.j 0.+0.j]
H|0>  = [0.70710678+0.j 0.70710678+0.j]
Ry(pi)|0> = [0.+0.j 1.+0.j]


In [5]:
I = np.eye(2, dtype=complex)   # birim matris (identity): hiçbir şey yapma

def uygula(durum, kapi, hedef):
    """2 kübitlik duruma tek kübitlik kapıyı uygular.
    hedef = 0 → q0 (soldaki), hedef = 1 → q1 (sağdaki)"""
    if hedef == 0:
        buyuk = np.kron(kapi, I)
    else:
        buyuk = np.kron(I, kapi)
    return buyuk @ durum

# Deneme 1: |00⟩'ın q0'ına H
print(np.round(uygula(ket00, H, 0), 3))

# Deneme 2: |00⟩'ın q1'ine X
print(np.round(uygula(ket00, X, 1), 3))

[0.707+0.j 0.   +0.j 0.707+0.j 0.   +0.j]
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]


In [8]:
# CNOT: kontrol q0 (soldaki), hedef q1 (sağdaki)
CNOT = np.array([[1, 0, 0, 0],
                 [0, 1, 0, 0],
                 [0, 0, 0, 1],
                 [0, 0, 1, 0]], dtype=complex)


for isim, ket in [("00", ket00), ("01", ket01), ("10", ket10), ("11", ket11)]:
    print(isim, "->", np.round(CNOT @ ket, 3).real)

00 -> [1. 0. 0. 0.]
01 -> [0. 1. 0. 0.]
10 -> [0. 0. 0. 1.]
11 -> [0. 0. 1. 0.]


In [10]:
def olc(durum, shots=1000):
    olasilik = np.abs(durum) ** 2            # Born kuralı: |genlik|²
    n = int(np.log2(len(durum)))             # kübit sayısı (4 kutu → 2 kübit)
    etiketler = [format(i, f"0{n}b") for i in range(len(durum))]   # "00","01","10","11"
    sonuclar = np.random.choice(etiketler, size=shots, p=olasilik)  # ağırlıklı zar
    sayim = {e: int(np.sum(sonuclar == e)) for e in etiketler}
    return sayim

# Deneme: |00⟩'ın q0'ına H → 1/√2 (|00⟩ + |10⟩)
durum = uygula(ket00, H, 0)
print(olc(durum))

{'00': 504, '01': 0, '10': 496, '11': 0}


In [12]:
# Test 1: H·H|0⟩ = |0⟩
sonuc = H @ (H @ ket0)        # önce içteki H, sonra dıştaki (sağdan sola)
print("H·H|0> =", np.round(sonuc, 3))

assert np.allclose(sonuc, ket0), "Test 1 BAŞARISIZ"
print("Test 1 geçti ✅")

H·H|0> = [1.+0.j 0.+0.j]
Test 1 geçti ✅


In [13]:
# Test 2: Bell devresi
bell = ket00                    # başlangıç |00⟩
bell = uygula(bell, H, 0)       # q0'a H   → 1/√2 (|00⟩ + |10⟩)
bell = CNOT @ bell              # CNOT     → 1/√2 (|00⟩ + |11⟩)
print("Bell durumu:", np.round(bell, 3).real)

sayim = olc(bell, shots=1000)
print("Ölçüm:", sayim)

assert sayim["01"] == 0 and sayim["10"] == 0, "Test 2 BAŞARISIZ"
print("Test 2 geçti ✅")

Bell durumu: [0.707 0.    0.    0.707]
Ölçüm: {'00': 474, '01': 0, '10': 0, '11': 526}
Test 2 geçti ✅
